# Prepare environment

In [1]:
%load_ext autoreload
%autoreload 2

## Init base libs

* Install libs

In [2]:
%%capture
%pip install \
    python-dotenv

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()
print(os.getenv('ELASTICSEARCH_URL', 'http://elasticsearch:9200'))

http://elasticsearch:9200


## Init ElasticSearch

In [4]:
%%capture
%pip install \
    elasticsearch==8.15.1

In [5]:
import elasticsearch
from elasticsearch import Elasticsearch
from elasticsearch import helpers

# values come from ../.env through docker-compose.yml
es_url = os.getenv('ELASTICSEARCH_URL', 'http://elasticsearch:9200')
es_user = os.getenv('ELASTIC_USER')
es_password = os.getenv('ELASTIC_PASSWORD')

# credentials only when ES security is enabled (xpack.security.enabled=false in this stack)
basic_auth = (es_user, es_password) if es_user else None

es = Elasticsearch(hosts=es_url, basic_auth=basic_auth)
print("Connection to", es_url, "as", es_user or "anonymous")
es.info() # should return cluster info

Connection to http://elasticsearch:9200 as anonymous


ObjectApiResponse({'name': 'ElasticSearch', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'Y6BQGVV3Roqwt3iNUi3OLQ', 'version': {'number': '8.15.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'f97532e680b555c3a05e73a74c28afb666923018', 'build_date': '2024-10-09T22:08:00.328917561Z', 'build_snapshot': False, 'lucene_version': '9.11.1', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

# Usage

Browse every index of the cluster, for example `customers` written by the Flink job
`flink-cdc-es` (one document per row of Postgres `inventory.customers`, `_id` = customer id).

In [6]:
import pandas as pd

# Settings for the cells below
INCLUDE_SYSTEM_INDEXES = False  # True to also show indexes starting with '.'
MAX_DOCS_PER_INDEX = None       # None = all documents, or a number to cap each listing

## List all indexes

Sorted by name, with health, status, document count and size.

In [7]:
def list_indexes():
    """Names of the indexes to browse, sorted."""
    rows = es.cat.indices(format="json", h="index,health,status,docs.count,store.size", s="index")
    return [r for r in rows if INCLUDE_SYSTEM_INDEXES or not r["index"].startswith(".")]

indexes = list_indexes()
pd.DataFrame(indexes)

,index,health,status,docs.count,store.size
0,accounts_connect,green,open,13,33.7kb
1,accounts_connect_demo,green,open,4,13.9kb
2,contacts_connect,green,open,16,13.5kb
3,customers,yellow,open,15,26.6kb
4,customers_connect,green,open,14,39.8kb
5,customers_connect_demo,green,open,8,17.8kb
6,customers_flinkapp,yellow,open,2,17.6kb


## Count documents of each index

Uses the count API: the exact number of searchable documents (`_cat/indices` can include
deleted or not yet refreshed documents).

In [8]:
counts = pd.DataFrame(
    [{"index": i["index"], "count": es.count(index=i["index"])["count"]} for i in indexes]
)
display(counts)
print("total:", counts["count"].sum() if not counts.empty else 0)

,index,count
0,accounts_connect,13
1,accounts_connect_demo,4
2,contacts_connect,16
3,customers,15
4,customers_connect,14
5,customers_connect_demo,8
6,customers_flinkapp,2


total: 72


## List all documents of each index

`helpers.scan` pages through each index with the scroll API, so it isn't capped at the
10 000 hits of a single search. Each row is one document: `_id` plus its `_source` fields.

In [9]:
def list_documents(index, max_docs=MAX_DOCS_PER_INDEX):
    """All documents of `index` as a DataFrame (`_id` + `_source` fields)."""
    docs = []
    for hit in helpers.scan(es, index=index, query={"query": {"match_all": {}}}):
        docs.append({"_id": hit["_id"], **hit["_source"]})
        if max_docs is not None and len(docs) >= max_docs:
            break
    df = pd.DataFrame(docs)
    if not df.empty:
        df = df.sort_values("_id", key=lambda ids: pd.to_numeric(ids, errors="coerce").fillna(float("inf")))
        df = df.reset_index(drop=True)
    return df

documents = {}
for i in indexes:
    name = i["index"]
    documents[name] = list_documents(name)
    print(f"\n=== {name}: {len(documents[name])} document(s)")
    display(documents[name])


=== accounts_connect: 13 document(s)


,_id,base_currency,cdc_source_ts,cdc_op,aliases,notes,account_ref,display_name,cdc_processed_ts,credit_currency,country_code,opened_at,owner_customer_id,credit_limit,id,legal_name,payment_terms_days,status
0,50001,USD,2026-11-09T07:15:46.235Z,c,None,"Monthly invoice, paid by bank transfer",acct_OAK001,Oakwood Home,2026-11-09T07:15:46.522Z,USD,US,2023-03-01T09:00:00Z,1001,10000.0,50001,Oakwood Home Supplies LLC,30,ACTIVE
1,50002,USD,2026-11-09T07:15:46.235Z,c,None,None,acct_DAV001,None,2026-11-09T07:15:46.522Z,USD,US,2023-03-02T09:00:00Z,1002,0.0,50002,Davies Design Studio,15,PENDING
2,50003,EUR,2026-11-09T07:15:46.262Z,c,None,Pipe organ restoration for two churches,acct_W1L0W5,Willow Organs,2026-11-09T07:15:46.526Z,EUR,FR,2024-03-20T09:00:00Z,1003,250.0,50003,Willow Organ Builders SARL,15,ACTIVE
3,50004,SGD,2026-11-09T07:15:46.262Z,c,[MPS],Ordering paused until the warehouses reopen,acct_K3S7R1,Meridian Pacific,2026-11-09T07:15:46.527Z,SGD,SG,2024-07-01T09:00:00Z,1004,2500.0,50004,Meridian Pacific Shipping Ltd,45,ACTIVE
4,50005,EUR,2026-11-09T07:15:46.262Z,c,None,"Organic produce wholesaler, awaiting documents",acct_GRN001,None,2026-11-09T07:15:46.527Z,EUR,DE,2025-01-15T09:00:00Z,1005,0.0,50005,Greenleaf Organic Produce GmbH,0,PENDING
5,50006,EUR,2026-11-09T07:15:46.262Z,c,None,Invoices disputed after the merger,acct_7Q2M9X,Kestrel,2026-11-09T07:15:46.527Z,EUR,DE,2024-01-15T09:00:00Z,1007,2500.0,50006,Kestrel Labs GmbH,30,ACTIVE
6,50007,GBP,2026-11-09T07:15:46.262Z,c,[NWT],Refunds delayed while audits are running,acct_7Q2M9K,Northwind,2026-11-09T07:15:46.527Z,GBP,GB,2023-05-02T09:00:00Z,1008,15000.0,50007,Northwind Traders Ltd,30,ACTIVE
7,50008,USD,2026-11-09T07:15:46.262Z,c,None,"Organization restructured last quarter, cards ...",acct_OKF001,Okafor Logistics,2026-11-09T07:15:46.527Z,USD,NG,2025-05-08T09:00:00Z,1009,5000.0,50008,Okafor & Sons Logistics Inc,30,SUSPENDED
8,50009,CNY,2026-11-09T07:15:46.262Z,c,[SLT],"Audit completed, the auditor requested statements",acct_SLT001,None,2026-11-09T07:15:46.527Z,JPY,CN,2025-07-06T09:00:00Z,1010,5000000.0,50009,Shanghai Lantern Trading Co Ltd,60,ACTIVE
9,50010,NOK,2026-11-09T07:15:46.262Z,c,None,Account closed after the season,acct_FJD001,None,2026-11-09T07:15:46.528Z,NOK,NO,2022-12-13T09:00:00Z,1011,75000.0,50010,Fjord Fisheries AS,60,CLOSED



=== accounts_connect_demo: 4 document(s)


,_id,base_currency,aliases,notes,account_ref,display_name,credit_currency,country_code,opened_at,owner_customer_id,credit_limit,id,legal_name,payment_terms_days,status
0,9101,GBP,[NWT],Refunds delayed while audits are running,acct_7Q2M9K,Northwind,GBP,GB,2023-05-02T09:00:00Z,9005,15000.0,9101,Northwind Traders Ltd,30,ACTIVE
1,9102,EUR,None,Invoices disputed after the merger,acct_7Q2M9X,Kestrel,EUR,DE,2024-01-15T09:00:00Z,9004,2500.0,9102,Kestrel Labs GmbH,30,ACTIVE
2,9103,SGD,[MPS],Ordering paused until the warehouses reopen,acct_K3S7R1,Meridian Pacific,SGD,SG,2024-07-01T09:00:00Z,9001,2500.0,9103,Meridian Pacific Shipping Ltd,45,ACTIVE
3,9104,EUR,None,Weekly flour orders,acct_DRU001,Dante's,EUR,IT,2025-10-11T09:00:00Z,9002,1000.0,9104,Dante's Pizzeria S.r.l.,15,ACTIVE



=== contacts_connect: 16 document(s)


,_id,cdc_source_ts,cdc_op,verified_at,phone,kind,id,label,customer_id,value,cdc_processed_ts,preferred,email
0,10001,2026-11-09T07:15:46.219Z,c,2023-02-18T09:05:00Z,(800) 555‑2100,PHONE,10001,WORK,1001,(800) 555‑2100,2026-11-09T07:15:46.516Z,True,NaN
1,10002,2026-11-09T07:15:46.219Z,c,2023-02-18T09:00:00Z,NaN,EMAIL,10002,WORK,1001,brandon.thomas@example.com,2026-11-09T07:15:46.517Z,True,brandon.thomas@example.com
2,10003,2026-11-09T07:15:46.219Z,c,None,(800) 555‑2300,PHONE,10003,WORK,1002,(800) 555‑2300,2026-11-09T07:15:46.517Z,False,NaN
3,10004,2026-11-09T07:15:46.219Z,c,None,(800) 555‑2200,PHONE,10004,HOME,1002,(800) 555‑2200,2026-11-09T07:15:46.517Z,False,NaN
4,10005,2026-11-09T07:15:46.219Z,c,None,(800) 555‑2100,PHONE,10005,MOBILE,1002,(800) 555‑2100,2026-11-09T07:15:46.517Z,True,NaN
5,10006,2026-11-09T07:15:46.219Z,c,2023-02-18T10:00:00Z,NaN,EMAIL,10006,WORK,1002,sarah.davies@example.com,2026-11-09T07:15:46.517Z,True,sarah.davies@example.com
6,10007,2026-11-09T07:15:46.253Z,c,None,NaN,EMAIL,10007,HOME,1003,mc.dubois@example.fr,2026-11-09T07:15:46.524Z,False,mc.dubois@example.fr
7,10008,2026-11-09T07:15:46.253Z,c,None,(415) 555-0123,PHONE,10008,WORK,1007,(415) 555-0123,2026-11-09T07:15:46.525Z,False,NaN
8,10009,2026-11-09T07:15:46.253Z,c,2023-09-10T10:00:00Z,NaN,EMAIL,10009,WORK,1007,a.petrova@northwind-traders.example,2026-11-09T07:15:46.525Z,True,a.petrova@northwind-traders.example
9,10010,2026-11-09T07:15:46.253Z,c,2025-07-06T03:00:00Z,+86 138 0013 8000,PHONE,10010,MOBILE,1010,+86 138 0013 8000,2026-11-09T07:15:46.525Z,True,NaN


... (4 outputs left out)


=== customers_connect_demo: 8 document(s)


,_id,country_code,full_name,status_code,registered_at,city,phone,last_name,id,customer_ref,first_name,email,status
0,9001,FR,Marie-Claire Dubois,1,2024-03-18T09:30:00Z,Lyon,+33 4 78 37 00 00,Dubois,9001,cus_mcd2403,Marie-Claire,marie-claire.dubois@example.fr,ACTIVE
1,9002,DE,Zoë Müller,0,2025-01-14T10:00:00Z,Berlin,+49 30 1234 5678,Müller,9002,cus_zmu2501,Zoë,zoe.mueller@example.de,PENDING
2,9003,IE,Siobhán D'Arcy,1,2024-11-02T15:20:00Z,Dublin,+353 1 555 0142,D'Arcy,9003,cus_sda2411,Siobhán,siobhan.darcy@example.ie,ACTIVE
3,9004,BG,Alexandra Petrova,1,2023-09-09T12:00:00Z,Sofia,+359 2 555 0175,Petrova,9004,cus_ape2309,Alexandra,alexandra.petrova@example.bg,ACTIVE
4,9005,PL,Tomasz Kowalski,2,2023-04-21T07:45:00Z,Warsaw,+48 22 555 0190,Kowalski,9005,cus_tko2304,Tomasz,tomasz.kowalski@example.pl,SUSPENDED
5,9006,NG,Chidi Okafor,1,2025-05-07T23:30:00Z,Lagos,+234 1 555 0188,Okafor,9006,cus_cok2505,Chidi,chidi.okafor@example.ng,ACTIVE
6,9007,CN,Wei Li,1,2025-07-05T11:00:00Z,Shanghai,+86 21 5555 0100,Li,9007,cus_wli2507,Wei,wei.li@example.cn,ACTIVE
7,9008,SE,Søren Lindqvist,1,2024-08-08T08:08:00Z,Stockholm,+46 8 555 0133,Lindqvist,9008,cus_sli2408,Søren,soren.lindqvist@example.se,ACTIVE



=== customers_flinkapp: 2 document(s)


,_id,id,first_name,last_name,full_name,email,street,zip,city,phone,status,registered,cdc_op,cdc_source_ts,cdc_processed_ts
0,1001,1001,Brandon,Thomas,Brandon Thomas,brandon.thomas@example.com,29 Oakwood Drive,90210,Los Angeles,(800) 555‑2100,1,2023-02-17T21:36:42Z,r,2026-11-08T15:57:12.211Z,2026-11-08T15:57:12.301Z
1,1002,1002,Sarah,Davies,Sarah Davies,sarah.davies@example.com,29 Oakwood Drive,90210,Los Angeles,NaN,1,2023-02-17T21:36:42Z,r,2026-11-08T15:57:12.211Z,2026-11-08T15:57:12.303Z


## Check

Every document of each index was read: the listing has as many rows as the count
(skipped when `MAX_DOCS_PER_INDEX` caps the listing).

In [10]:
for _, row in counts.iterrows():
    listed = len(documents[row["index"]])
    if MAX_DOCS_PER_INDEX is None:
        assert listed == row["count"], f"{row['index']}: count={row['count']} but listed={listed}"
    print(f"OK {row['index']}: count={row['count']} listed={listed}")

OK accounts_connect: count=13 listed=13
OK accounts_connect_demo: count=4 listed=4
OK contacts_connect: count=16 listed=16
OK customers: count=15 listed=15
OK customers_connect: count=14 listed=14
OK customers_connect_demo: count=8 listed=8
OK customers_flinkapp: count=2 listed=2


# Search: blur and fuzzy

The inventory model (`../model`, see `model/README.md`) opts each field in to search behaviors
that are independent of each other:

| Behavior | Tolerates | Where | How |
|---|---|---|---|
| blur | case, accents, hyphen / apostrophe parts, word order, legal suffixes | names, aliases, email fragments | analyzers of the component template `inventory-text-analysis`, always on |
| prefix | the start of each word, from 3 characters | names (`.prefix`) | edge n-grams, always on |
| fuzzy | 1 typo from 3 characters (`AUTO:3,64`) | names only, never aliases, references, contacts | `fuzzy=true` in the request only |

Every clause of a search template is named, so `matched_queries` says why a hit was found
(`name_exact`, `name_words`, `name_prefix`, `email_fragment`, `alias`, `name_fuzzy`...).

The cells below use **demo indices** `customers_connect_demo` and `accounts_connect_demo`: they
match the index template patterns (`customers_connect*`, `accounts_connect*`), so they get the
same analyzers, mapping and default pipeline as the CDC indices, but the documents inserted here
never mix with the rows Debezium writes (`customers_connect` must stay equal to Postgres).

In [11]:
import json
from pathlib import Path
from elasticsearch import NotFoundError

# ../model, mounted read-only at /domain in the jupyter container (docker-compose.yml)
MODEL_DIR = Path(os.getenv("DOMAIN_DIR", "/domain"))
ES_DIR = MODEL_DIR / "elasticsearch"
DEMO_SUFFIX = "_demo"  # customers_connect_demo matches the index template pattern customers_connect*


def load(name):
    return json.loads((ES_DIR / name).read_text())


metadata = load("auto_apply_metadata.json")
search_template_by_index = {s["index"]: s for s in metadata["search_templates"]}

DEMO = {
    "customer": {"index": "customers_connect" + DEMO_SUFFIX, "key": "customer_ref",
                 "search_template": search_template_by_index["customers_connect"]["name"]},
    "account": {"index": "accounts_connect" + DEMO_SUFFIX, "key": "account_ref",
                "search_template": search_template_by_index["accounts_connect"]["name"]},
}
pd.DataFrame(DEMO).T

,index,key,search_template
customer,customers_connect_demo,customer_ref,customers-general
account,accounts_connect_demo,account_ref,accounts-general


## Create the definitions when they don't exist

From `model/elasticsearch` (names in `auto_apply_metadata.json`), in dependency order: component
template (analyzers), ingest pipelines, index templates, demo indices, search templates. Anything
that already exists is left unchanged: the CDC notebook (or `model/tools/apply_elasticsearch.py`)
puts the latest version of the files. The live mapping of each demo index must then be the one of
the file.

In [12]:
def exists(call, **kwargs):
    """True when a GET of the definition succeeds."""
    try:
        call(**kwargs)
        return True
    except NotFoundError:
        return False


def ensure(kind, name, found, create):
    if found:
        print(f"{kind} {name}: exists, left unchanged")
    else:
        create()
        print(f"{kind} {name}: created")


for c in metadata["component_templates"]:
    body = load(c["filename"])
    ensure("component template", c["name"], es.cluster.exists_component_template(name=c["name"]).body,
           lambda: es.cluster.put_component_template(name=c["name"], template=body["template"], meta=body.get("_meta")))

for p in metadata["ingest_pipelines"]:
    body = load(p["filename"])
    ensure("ingest pipeline", p["name"], exists(es.ingest.get_pipeline, id=p["name"]),
           lambda: es.ingest.put_pipeline(id=p["name"], description=body["description"],
                                          processors=body["processors"], meta=body.get("_meta")))

for t in metadata["index_templates"]:
    body = load(t["filename"])
    ensure("index template", t["name"], es.indices.exists_index_template(name=t["name"]).body,
           lambda: es.indices.put_index_template(
               name=t["name"], index_patterns=t["index_patterns"], composed_of=t.get("composed_of", []),
               template={"settings": body["settings"], "mappings": body["mappings"]},
               meta={"source": f"model/elasticsearch/{t['filename']}"}))

for d in DEMO.values():
    # no body: settings, analyzers, mapping and default_pipeline come from the index template
    ensure("index", d["index"], es.indices.exists(index=d["index"]).body, lambda: es.indices.create(index=d["index"]))

for s in metadata["search_templates"]:
    ensure("search template", s["name"], exists(es.get_script, id=s["name"]),
           lambda: es.put_script(id=s["name"], script={"lang": "mustache", "source": (ES_DIR / s["filename"]).read_text()}))

component template inventory-text-analysis: exists, left unchanged
ingest pipeline customers-cdc: exists, left unchanged
ingest pipeline contacts-cdc: exists, left unchanged
ingest pipeline accounts-cdc: exists, left unchanged
ingest pipeline inventory-cdc-common: exists, left unchanged
index template inventory-customers: exists, left unchanged
index template inventory-contacts: exists, left unchanged
index template inventory-accounts: exists, left unchanged
index customers_connect_demo: exists, left unchanged
index accounts_connect_demo: exists, left unchanged
search template customers-general: exists, left unchanged
search template contacts-lookup: exists, left unchanged
search template accounts-general: exists, left unchanged


In [13]:
def normalize(mapping):
    """ES returns `dynamic` as a string: compare values only."""
    return {"dynamic": str(mapping.get("dynamic", "true")).lower(), "properties": mapping["properties"]}


file_by_index = {t["index"]: t["filename"] for t in metadata["index_templates"]}
for entity, d in DEMO.items():
    file_mapping = load(file_by_index[d["index"].removesuffix(DEMO_SUFFIX)])["mappings"]
    live_mapping = es.indices.get_mapping(index=d["index"])[d["index"]]["mappings"]
    assert normalize(live_mapping) == normalize(file_mapping), \
        f"{d['index']} mapping differs from the file: delete the demo index and run the cell above again"
    pipeline = es.indices.get_settings(index=d["index"])[d["index"]]["settings"]["index"].get("default_pipeline")
    print(f"OK {d['index']}: mapping of the file, default_pipeline {pipeline}")

OK customers_connect_demo: mapping of the file, default_pipeline customers-cdc
OK accounts_connect_demo: mapping of the file, default_pipeline accounts-cdc


## Blur: what the analyzers make of a value

Indexed values are split and folded at index time (`*_index_analyzer`); the query goes through
the matching `*_search_analyzer`. A hit needs a common token, so these tokens explain every
blur case below: `marie-claire` is kept whole *and* split, but never joined into `marieclaire`;
`ü` is folded to `u`, not transliterated to `ue`; legal suffixes disappear on both sides.

In [14]:
analyzer_samples = [
    ("customer", "name_index_analyzer", "Marie-Claire D'Arcy"),
    ("customer", "name_search_analyzer", "ZOË Müller"),
    ("customer", "name_prefix_index_analyzer", "Alexandra"),
    ("customer", "email_text_analyzer", "kai_lee+invoices.2025@mailbox.example"),
    ("account", "company_index_analyzer", "Kestrel Labs GmbH"),
    ("account", "company_search_analyzer", "GMBH KESTREL"),
    ("account", "english_analyzer", "Refunds delayed while audits are running"),
]
pd.DataFrame([
    {"analyzer": analyzer, "text": text_,
     "tokens": [t["token"] for t in es.indices.analyze(index=DEMO[entity]["index"], analyzer=analyzer, text=text_)["tokens"]]}
    for entity, analyzer, text_ in analyzer_samples
])

,analyzer,text,tokens
0,name_index_analyzer,Marie-Claire D'Arcy,"[marie-claire, marie, claire, d'arcy, d, arcy]"
1,name_search_analyzer,ZOË Müller,"[zoe, muller]"
2,name_prefix_index_analyzer,Alexandra,"[ale, alex, alexa, alexan, alexand, alexandr, ..."
3,email_text_analyzer,kai_lee+invoices.2025@mailbox.example,"[kai, lee, invoices, 2025, mailbox, example]"
4,company_index_analyzer,Kestrel Labs GmbH,"[kestrel, labs]"
5,company_search_analyzer,GMBH KESTREL,[kestrel]
6,english_analyzer,Refunds delayed while audits are running,"[refund, delai, while, audit, run]"


## Insert sample documents

Rows shaped like the ones Debezium sends (column names, `status` code, `credit_limit_minor` in
minor units), so the index `default_pipeline` builds the same document as for the CDC indices
(`full_name`, `status` label, `credit_limit` in major units). Values from
`model/migration/V21__insert_customers_variants.sql` and `V23__insert_accounts_variants.sql`. Each
document has a fixed `_id`: running the cell again overwrites them.

In [15]:
CUSTOMER_COLUMNS = ["id", "customer_ref", "first_name", "last_name", "email", "city", "country_code", "phone", "status", "registered_at"]
customer_rows = [
    (9001, "cus_mcd2403", "Marie-Claire", "Dubois", "marie-claire.dubois@example.fr", "Lyon", "FR", "+33 4 78 37 00 00", 1, "2024-03-18T09:30:00Z"),
    (9002, "cus_zmu2501", "Zoë", "Müller", "zoe.mueller@example.de", "Berlin", "DE", "+49 30 1234 5678", 0, "2025-01-14T10:00:00Z"),
    (9003, "cus_sda2411", "Siobhán", "D'Arcy", "siobhan.darcy@example.ie", "Dublin", "IE", "+353 1 555 0142", 1, "2024-11-02T15:20:00Z"),
    (9004, "cus_ape2309", "Alexandra", "Petrova", "alexandra.petrova@example.bg", "Sofia", "BG", "+359 2 555 0175", 1, "2023-09-09T12:00:00Z"),
    (9005, "cus_tko2304", "Tomasz", "Kowalski", "tomasz.kowalski@example.pl", "Warsaw", "PL", "+48 22 555 0190", 2, "2023-04-21T07:45:00Z"),
    (9006, "cus_cok2505", "Chidi", "Okafor", "chidi.okafor@example.ng", "Lagos", "NG", "+234 1 555 0188", 1, "2025-05-07T23:30:00Z"),
    (9007, "cus_wli2507", "Wei", "Li", "wei.li@example.cn", "Shanghai", "CN", "+86 21 5555 0100", 1, "2025-07-05T11:00:00Z"),
    (9008, "cus_sli2408", "Søren", "Lindqvist", "soren.lindqvist@example.se", "Stockholm", "SE", "+46 8 555 0133", 1, "2024-08-08T08:08:00Z"),
]
ACCOUNT_COLUMNS = ["id", "account_ref", "owner_customer_id", "legal_name", "display_name", "aliases", "country_code",
                   "base_currency", "credit_currency", "credit_limit_minor", "payment_terms_days", "status", "notes", "opened_at"]
account_rows = [
    (9101, "acct_7Q2M9K", 9005, "Northwind Traders Ltd", "Northwind", ["NWT"], "GB", "GBP", None, 1500000, 30, "ACTIVE",
     "Refunds delayed while audits are running", "2023-05-02T09:00:00Z"),
    (9102, "acct_7Q2M9X", 9004, "Kestrel Labs GmbH", "Kestrel", None, "DE", "EUR", None, 250000, 30, "ACTIVE",
     "Invoices disputed after the merger", "2024-01-15T09:00:00Z"),
    (9103, "acct_K3S7R1", 9001, "Meridian Pacific Shipping Ltd", "Meridian Pacific", ["MPS"], "SG", "SGD", None, 250000, 45, "ACTIVE",
     "Ordering paused until the warehouses reopen", "2024-07-01T09:00:00Z"),
    (9104, "acct_DRU001", 9002, "Dante's Pizzeria S.r.l.", "Dante's", None, "IT", "EUR", None, 100000, 15, "ACTIVE",
     "Weekly flour orders", "2025-10-11T09:00:00Z"),
]

samples = {"customer": [dict(zip(CUSTOMER_COLUMNS, r)) for r in customer_rows],
           "account": [dict(zip(ACCOUNT_COLUMNS, r)) for r in account_rows]}
actions = [{"_index": DEMO[entity]["index"], "_id": row["id"], "_source": row}
           for entity, rows in samples.items() for row in rows]
ok, errors = helpers.bulk(es, actions, refresh="wait_for")
assert not errors, errors
print(f"{ok} document(s) indexed")

for entity, d in DEMO.items():
    hits = es.search(index=d["index"], size=100, sort=["id"])["hits"]["hits"]
    assert {h["_id"] for h in hits} >= {str(r["id"]) for r in samples[entity]}, f"{d['index']}: missing documents"
    fields = ["full_name", "email", "status", "status_code"] if entity == "customer" else \
             ["legal_name", "aliases", "credit_currency", "credit_limit", "notes"]
    print(f"\n=== {d['index']}: {len(hits)} document(s), after the pipeline")
    display(pd.DataFrame([{"_id": h["_id"], d["key"]: h["_source"][d["key"]], **{f: h["_source"].get(f) for f in fields}} for h in hits]))

12 document(s) indexed

=== customers_connect_demo: 8 document(s), after the pipeline


,_id,customer_ref,full_name,email,status,status_code
0,9001,cus_mcd2403,Marie-Claire Dubois,marie-claire.dubois@example.fr,ACTIVE,1
1,9002,cus_zmu2501,Zoë Müller,zoe.mueller@example.de,PENDING,0
2,9003,cus_sda2411,Siobhán D'Arcy,siobhan.darcy@example.ie,ACTIVE,1
3,9004,cus_ape2309,Alexandra Petrova,alexandra.petrova@example.bg,ACTIVE,1
4,9005,cus_tko2304,Tomasz Kowalski,tomasz.kowalski@example.pl,SUSPENDED,2
5,9006,cus_cok2505,Chidi Okafor,chidi.okafor@example.ng,ACTIVE,1
6,9007,cus_wli2507,Wei Li,wei.li@example.cn,ACTIVE,1
7,9008,cus_sli2408,Søren Lindqvist,soren.lindqvist@example.se,ACTIVE,1



=== accounts_connect_demo: 4 document(s), after the pipeline


,_id,account_ref,legal_name,aliases,credit_currency,credit_limit,notes
0,9101,acct_7Q2M9K,Northwind Traders Ltd,[NWT],GBP,15000.0,Refunds delayed while audits are running
1,9102,acct_7Q2M9X,Kestrel Labs GmbH,None,EUR,2500.0,Invoices disputed after the merger
2,9103,acct_K3S7R1,Meridian Pacific Shipping Ltd,[MPS],SGD,2500.0,Ordering paused until the warehouses reopen
3,9104,acct_DRU001,Dante's Pizzeria S.r.l.,None,EUR,1000.0,Weekly flour orders


## Test blur and fuzzy search

Each case runs the stored search template (`customers-general`, `accounts-general`) on a demo
index: `(entity, behavior, case, params, found, not_found, exact)`, as in `model/tests`
(`support.SearchTestCase.run_cases`):

* `found` `{key: named query}`: must be a hit, explained by that named query (`None`: any)
* `not_found` `[keys]` must not be a hit, or `{key: [named queries]}` must not be explained by them
* `exact`: the hits are exactly the keys of `found`

Fuzzy matching is off unless the request sets `fuzzy`; `fuzziness` overrides `AUTO:3,64`.

In [16]:
NAME_TIERS = ["name_exact", "name_words", "name_prefix", "name_fuzzy"]


def search(entity, params):
    """Hits of the stored search template, in rank order: {key: sorted matched_queries}."""
    d = DEMO[entity]
    result = es.search_template(index=d["index"], id=d["search_template"], params={"size": 50, **params})
    return {h["_source"][d["key"]]: sorted(h.get("matched_queries", [])) for h in result["hits"]["hits"]}


def check(hits, found, not_found, exact):
    """Problems of one case, empty when it passes."""
    problems = []
    for key, tier in found.items():
        if key not in hits:
            problems.append(f"{key} not found")
        elif tier and tier not in hits[key]:
            problems.append(f"{key} matched by {hits[key]}, not {tier}")
    if isinstance(not_found, dict):
        problems += [f"{k} matched by {sorted(set(hits.get(k, [])) & set(t))}" for k, t in not_found.items() if set(hits.get(k, [])) & set(t)]
    else:
        problems += [f"{k} found" for k in not_found if k in hits]
    if exact and set(hits) != set(found):
        problems.append(f"unexpected hits {sorted(set(hits) - set(found))}")
    return problems


CASES = [
    # customers-general: person names
    ("customer", "blur", "hyphen part + surname", {"query": "claire dubois"}, {"cus_mcd2403": "name_words"}, [], True),
    ("customer", "blur", "reordered, upper case", {"query": "DUBOIS MARIE"}, {"cus_mcd2403": "name_words"}, [], True),
    ("customer", "blur", "parts are never joined", {"query": "marieclaire"}, {}, ["cus_mcd2403"], True),
    ("customer", "blur", "accents folded", {"query": "zoe muller"}, {"cus_zmu2501": "name_exact"}, [], True),
    ("customer", "blur", "accented query", {"query": "ZOË"}, {"cus_zmu2501": "name_words"}, [], False),
    ("customer", "blur", "ø folded", {"query": "soren"}, {"cus_sli2408": "name_words"}, [], False),
    ("customer", "blur", "ü not transliterated: email only", {"query": "mueller"}, {"cus_zmu2501": "email_fragment"}, {"cus_zmu2501": NAME_TIERS}, True),
    ("customer", "blur", "apostrophe part", {"query": "arcy"}, {"cus_sda2411": "name_words"}, [], True),
    ("customer", "blur", "apostrophe kept", {"query": "D'ARCY"}, {"cus_sda2411": "name_words"}, [], True),
    ("customer", "blur", "apostrophe dropped: email only", {"query": "darcy"}, {"cus_sda2411": "email_fragment"}, {"cus_sda2411": NAME_TIERS}, True),
    ("customer", "blur", "no nickname", {"query": "Sasha"}, {}, [], True),
    ("customer", "prefix", "every word is a prefix", {"query": "alex petr"}, {"cus_ape2309": "name_prefix"}, [], True),
    ("customer", "prefix", "under 3 characters", {"query": "al"}, {}, ["cus_ape2309"], False),
    ("customer", "fuzzy", "off by default", {"query": "Kowalsky"}, {}, ["cus_tko2304"], True),
    ("customer", "fuzzy", "substitution", {"query": "Kowalsky", "fuzzy": True}, {"cus_tko2304": "name_fuzzy"}, [], True),
    ("customer", "fuzzy", "insertion", {"query": "Kowalskii", "fuzzy": True}, {"cus_tko2304": "name_fuzzy"}, [], True),
    ("customer", "fuzzy", "deletion", {"query": "Okafr", "fuzzy": True}, {"cus_cok2505": "name_fuzzy"}, [], True),
    ("customer", "fuzzy", "transposition", {"query": "Okaofr", "fuzzy": True}, {"cus_cok2505": "name_fuzzy"}, [], True),
    ("customer", "fuzzy", "2 edits refused (AUTO:3,64)", {"query": "Kovalsky", "fuzzy": True}, {}, ["cus_tko2304"], True),
    ("customer", "fuzzy", "2 edits with ES AUTO", {"query": "Kovalsky", "fuzzy": True, "fuzziness": "AUTO"}, {"cus_tko2304": "name_fuzzy"}, [], True),
    ("customer", "fuzzy", "short word: no edit", {"query": "lu", "fuzzy": True}, {}, ["cus_wli2507"], True),
    ("customer", "fuzzy", "too far", {"query": "Okxxxx", "fuzzy": True}, {}, ["cus_cok2505"], True),
    # accounts-general: company names, aliases, notes
    ("account", "blur", "reordered words", {"query": "traders northwind"}, {"acct_7Q2M9K": "name_words"}, [], True),
    ("account", "blur", "legal suffix ignored", {"query": "GMBH KESTREL"}, {"acct_7Q2M9X": "name_words"}, [], True),
    ("account", "blur", "legal suffix alone", {"query": "ltd"}, {}, [], True),
    ("account", "blur", "possessive", {"query": "dante"}, {"acct_DRU001": "name_words"}, [], True),
    ("account", "blur", "maintained alias", {"query": "NWT"}, {"acct_7Q2M9K": "alias"}, [], True),
    ("account", "blur", "English word forms in notes", {"query": "refund delay"}, {"acct_7Q2M9K": "notes_words"}, [], True),
    ("account", "prefix", "every word is a prefix", {"query": "merid pac"}, {"acct_K3S7R1": "name_prefix"}, [], True),
    ("account", "fuzzy", "off by default", {"query": "Nortwind"}, {}, ["acct_7Q2M9K"], True),
    ("account", "fuzzy", "deletion", {"query": "Nortwind", "fuzzy": True}, {"acct_7Q2M9K": "name_fuzzy"}, [], True),
    ("account", "fuzzy", "substitution", {"query": "Kestral", "fuzzy": True}, {"acct_7Q2M9X": "name_fuzzy"}, [], True),
    ("account", "fuzzy", "transposition", {"query": "Meridain Pacific", "fuzzy": True}, {"acct_K3S7R1": "name_fuzzy"}, [], True),
    ("account", "fuzzy", "never on aliases", {"query": "NWX", "fuzzy": True}, {}, [], True),
    ("account", "fuzzy", "too far", {"query": "Northwxxx", "fuzzy": True}, {}, ["acct_7Q2M9K"], True),
]

results = []
for entity, behavior, case, params, found, not_found, exact in CASES:
    hits = search(entity, params)
    problems = check(hits, found, not_found, exact)
    results.append({"entity": entity, "behavior": behavior, "case": case, "params": params,
                    "hits": "; ".join(f"{k} {v}" for k, v in hits.items()) or "-",
                    "result": "FAIL: " + ", ".join(problems) if problems else "OK"})

results = pd.DataFrame(results)
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(results)

failed = results[results["result"] != "OK"]
assert failed.empty, f"{len(failed)} case(s) failed:\n{failed[['entity', 'case', 'result']].to_string()}"
print(f"OK: {len(results)} case(s)")

,entity,behavior,case,params,hits,result
0,customer,blur,hyphen part + surname,{'query': 'claire dubois'},"cus_mcd2403 ['email_fragment', 'name_exact', 'name_prefix', 'name_words']",OK
1,customer,blur,"reordered, upper case",{'query': 'DUBOIS MARIE'},"cus_mcd2403 ['email_fragment', 'name_prefix', 'name_words']",OK
2,customer,blur,parts are never joined,{'query': 'marieclaire'},-,OK
3,customer,blur,accents folded,{'query': 'zoe muller'},"cus_zmu2501 ['name_exact', 'name_prefix', 'name_words']",OK
4,customer,blur,accented query,{'query': 'ZOË'},"cus_zmu2501 ['email_fragment', 'name_exact', 'name_prefix', 'name_words']",OK
5,customer,blur,ø folded,{'query': 'soren'},"cus_sli2408 ['email_fragment', 'name_exact', 'name_prefix', 'name_words']",OK
6,customer,blur,ü not transliterated: email only,{'query': 'mueller'},cus_zmu2501 ['email_fragment'],OK
7,customer,blur,apostrophe part,{'query': 'arcy'},"cus_sda2411 ['name_exact', 'name_prefix', 'name_words']",OK
8,customer,blur,apostrophe kept,{'query': 'D'ARCY'},"cus_sda2411 ['name_exact', 'name_prefix', 'name_words']",OK
9,customer,blur,apostrophe dropped: email only,{'query': 'darcy'},cus_sda2411 ['email_fragment'],OK


OK: 35 case(s)


### Ranking: an exact match ranks before a typo

With `fuzzy=true` the typo tier is added at a low weight (x0.5), so a correctly spelled name still
comes first, explained by `name_exact` (x10). `_score` and `matched_queries` of each hit:

In [17]:
for entity, query, expected in [("customer", "Tomasz Kowalski", "cus_tko2304"), ("account", "Kestrel", "acct_7Q2M9X")]:
    d = DEMO[entity]
    result = es.search_template(index=d["index"], id=d["search_template"], params={"query": query, "fuzzy": True, "size": 5})
    hits = result["hits"]["hits"]
    display(pd.DataFrame([{"query": query, "_score": h["_score"], d["key"]: h["_source"][d["key"]],
                           "matched_queries": sorted(h.get("matched_queries", []))} for h in hits]))
    assert hits and hits[0]["_source"][d["key"]] == expected and "name_exact" in hits[0].get("matched_queries", []), \
        f"{query}: first hit {hits[0]['_source'][d['key']] if hits else None}"
    print(f"OK: {query} -> {expected} first, by name_exact")

,query,_score,customer_ref,matched_queries
0,Tomasz Kowalski,83.016884,cus_tko2304,"[email_fragment, name_exact, name_fuzzy, name_..."


OK: Tomasz Kowalski -> cus_tko2304 first, by name_exact


,query,_score,account_ref,matched_queries
0,Kestrel,26.886927,acct_7Q2M9X,"[name_exact, name_fuzzy, name_prefix, name_words]"


OK: Kestrel -> acct_7Q2M9X first, by name_exact


## Clean up

The demo indices only hold the documents above. To remove them (the definitions stay):

```python
for d in DEMO.values():
    es.indices.delete(index=d["index"], ignore_unavailable=True)
```